# 08. Експерименти з 1D-CNN

**Етап 08 з 13** · NASA C-MAPSS FD001 · RUL prediction


In [11]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## Налаштування

In [ ]:
import mlflow

mlflow.set_tracking_uri(
    "https://stage-broom-alkalize.ngrok-free.dev"
)

experiment = mlflow.set_experiment(
    "Predictive Maintenance - CNN and LSTM"
)

print("Experiment ID:", experiment.experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())

In [13]:
import copy
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DATA_DIR = Path(os.environ.get("FD001_DIR", "/content/drive/MyDrive/predictive_maintenance"))
CHECKPOINT_DIR = DATA_DIR / "checkpoints_model"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_PATH = DATA_DIR / "cnn" / "cnn_fd001_best.pt"

QUICK = os.environ.get("FD001_QUICK", "0") == "1"   # короткий тестовий прогін
SEED = 42
WINDOW_SIZE = 30
STRIDE = 1
SEEDS = [7, 21, 123, 2026]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| QUICK:", QUICK)

Device: cuda | QUICK: False


## 1. Дані

Розбиття за двигунами (`GroupShuffleSplit`, `test_size=0.2`, `random_state=42`), щоб один двигун не потрапляв і в train, і в validation.
Ця функція використовується і для сирого файлу FD001 (25 ознак), і для файлу з 49 ознаками.

In [14]:
COLUMNS = ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"] + [f"sensor_{i}" for i in range(1, 22)]
BASE_FEATURES = COLUMNS[1:]  # cycle, settings, sensors: 25 ознак


def split_by_engines(df):
    """Ділить рядки на train і validation за двигунами (unit_id), щоб двигуни не перетиналися."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(splitter.split(df, groups=df["unit_id"]))
    return df.iloc[train_idx].copy(), df.iloc[val_idx].copy()


# Сирі дані FD001: RUL = max(cycle) двигуна − поточний cycle
raw = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)
raw["RUL"] = raw.groupby("unit_id")["cycle"].transform("max") - raw["cycle"]
raw_train, raw_val = split_by_engines(raw)

# Файл з 49 ознаками (RUL уже є в файлі). Дубль колонки cycle.1 прибираємо, якщо він ідентичний cycle
feat_full = pd.read_csv(DATA_DIR / "new_train_49_features.csv")
if (feat_full["cycle"] == feat_full["cycle.1"]).all():
    feat_full = feat_full.drop(columns=["cycle.1"])
new_train, new_val = split_by_engines(feat_full)

print(f"Сирі: train {len(raw_train)} рядків / {raw_train['unit_id'].nunique()} двигунів, "
      f"val {len(raw_val)} / {raw_val['unit_id'].nunique()}")
print(f"49 ознак: train {len(new_train)} рядків, val {len(new_val)} рядків")

Сирі: train 16561 рядків / 80 двигунів, val 4070 / 20
49 ознак: train 13160 рядків, val 3401 рядків


### Датасет і вікна

Вікно — це `W` послідовних циклів одного двигуна. Ціль — RUL на останньому циклі вікна.

In [15]:
class RULDataset(Dataset):
    """Пара (вікно, RUL); вікно переставлене у формат (канали, довжина) для Conv1d."""

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).transpose(1, 2)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def create_windows(df, features, window_size, stride=1):
    """Формує вікна довжини `window_size` для кожного двигуна; повертає (X, y) з RUL останнього циклу."""
    X_windows, y_windows = [], []
    for _, engine in df.groupby("unit_id"):
        engine = engine.sort_values("cycle")
        X_engine = engine[features].to_numpy(dtype=np.float32)
        y_engine = engine["RUL"].to_numpy(dtype=np.float32)
        for start in range(0, len(engine) - window_size + 1, stride):
            end = start + window_size
            X_windows.append(X_engine[start:end])
            y_windows.append(y_engine[end - 1])
    return np.stack(X_windows), np.asarray(y_windows, dtype=np.float32)

## 2. Ознаки

У 49-ознаковому файлі: 25 базових, 7 wavelet `d2_std`, 1 `degradation_score`, 16 CTO v2 (разом 49).

| Набір | Склад | Ознак |
|---|---|---|
| `raw` | 25 базових (сирий FD001) | 25 |
| `A` | базові + `degradation_score` + індикатор його пропуску | 27 |
| `B` | усі 49 | 49 |
| `C` | усі 49 + індикатори пропусків (17 ознак з NaN) | 66 |

Пропуски заповнюються нулем, індикатори рахуються за train. Стандартизація — `StandardScaler`, fit лише на train; бінарні індикатори не стандартизуються.

In [ ]:
DEGRADATION_FEATURES = BASE_FEATURES + ["degradation_score"]
ALL_FEATURES = [c for c in feat_full.columns if c not in ["unit_id", "RUL", "cycle.1"]]

# набір -> (список ознак, чи додавати індикатори пропусків)
FEATURE_SETS = {
    "A": (DEGRADATION_FEATURES, True),
    "B": (ALL_FEATURES, False),
    "C": (ALL_FEATURES, True),
}


def prepare_feature_set(train_df, val_df, features, add_missing_indicators):
    """Додає індикатори пропусків (визначені за train) і заповнює NaN нулями; повертає фінальний список ознак."""
    train, val = train_df.copy(), val_df.copy()
    indicators = []
    if add_missing_indicators:
        for col in [c for c in features if train[c].isna().any()]:
            name = f"{col}_was_missing"
            train[name] = train[col].isna().astype("int8")
            val[name] = val[col].isna().astype("int8")
            indicators.append(name)

    train[features] = train[features].fillna(0)
    val[features] = val[features].fillna(0)
    final = features + indicators
    assert train[final].isna().sum().sum() == 0 and val[final].isna().sum().sum() == 0
    assert np.isfinite(train[final].to_numpy()).all() and np.isfinite(val[final].to_numpy()).all()
    return train, val, final


def scale_features(train_df, val_df, features):
    """Стандартизує числові ознаки за статистиками train; індикатори пропусків лишаються бінарними."""
    train, val = train_df.copy(), val_df.copy()
    numeric = [c for c in features if not c.endswith("_was_missing")]
    scaler = StandardScaler()
    train[numeric] = scaler.fit_transform(train[numeric])
    val[numeric] = scaler.transform(val[numeric])
    return train, val


_SPLIT_CACHE = {}


def build_split(data):
    """Повертає (train, val, features) після підготовки і стандартизації для набору `data` ("raw" або ключ FEATURE_SETS)."""
    if data not in _SPLIT_CACHE:
        if data == "raw":
            train, val, features = raw_train, raw_val, BASE_FEATURES
        else:
            cols, indicators = FEATURE_SETS[data]
            train, val, features = prepare_feature_set(new_train, new_val, cols, indicators)
        train, val = scale_features(train, val, features)
        _SPLIT_CACHE[data] = (train, val, features)
    return _SPLIT_CACHE[data]


_DATASET_CACHE = {}


def get_datasets(data, window_size):
    """Повертає (train_ds, val_ds, кількість каналів) для набору ознак і довжини вікна. Результат кешується."""
    key = (data, window_size)
    if key not in _DATASET_CACHE:
        train, val, features = build_split(data)
        X_tr, y_tr = create_windows(train, features, window_size, STRIDE)
        X_va, y_va = create_windows(val, features, window_size, STRIDE)
        print(f"[{data}, W={window_size}] train {X_tr.shape}, val {X_va.shape}")
        _DATASET_CACHE[key] = (RULDataset(X_tr, y_tr), RULDataset(X_va, y_va), len(features))
    return _DATASET_CACHE[key]


_, _, n_raw = get_datasets("raw", WINDOW_SIZE)
print("Ознак: raw", n_raw, "| A", len(build_split("A")[2]), "| B", len(build_split("B")[2]), "| C", len(build_split("C")[2]))

[raw, W=30] train (14241, 30, 25), val (3490, 30, 25)
Ознак: raw 25 | A 27 | B 49 | C 66


## 3. Експерименти

### Архітектура

Одна параметризована модель `RULCNN` замість окремих класів. Варіанти, що використовувались у запусках:

- `base`: Conv(25→32, k=3) → ReLU → MaxPool(2) → Conv(32→64, k=3) → ReLU → MaxPool(2) → FC 384→64
- `small`: те саме, але канали (16, 32); FC 192→64
- `nopool`: без MaxPool; FC 64·(W−4)→64

У всіх варіантах голова: Linear → ReLU → Dropout → Linear(1).

In [17]:
class RULCNN(nn.Module):
    """1D-CNN для регресії RUL. pooling="max" додає MaxPool1d(2) після кожного Conv1d, "none" — без стискання."""

    def __init__(self, in_channels, window_size, channels=(32, 64), kernel_size=3, pooling="max", dropout_p=0.0):
        super().__init__()
        layers, c_in, length = [], in_channels, window_size
        for c_out in channels:
            layers += [nn.Conv1d(c_in, c_out, kernel_size), nn.ReLU()]
            length -= kernel_size - 1
            if pooling == "max":
                layers.append(nn.MaxPool1d(2))
                length //= 2
            c_in = c_out
        self.conv_layers = nn.Sequential(*layers)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c_in * length, 64),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x)).squeeze(-1)


ARCHITECTURES = {
    "base": dict(channels=(32, 64), pooling="max"),
    "small": dict(channels=(16, 32), pooling="max"),
    "nopool": dict(channels=(32, 64), pooling="none"),
}

### Цикл навчання

In [18]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    """Один епоховий прохід по train; повертає середній MSE."""
    model.train()
    total, n = 0.0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * X.size(0)
        n += X.size(0)
    return total / n


def eval_loop(dataloader, model, loss_fn, device):
    """Оцінка на validation без градієнтів; повертає MSE, MAE і RMSE."""
    model.eval()
    total_loss = total_abs = total_sq = 0.0
    n = 0
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            total_loss += loss_fn(y_pred, y).item() * X.size(0)
            total_abs += (y_pred - y).abs().sum().item()
            total_sq += (y_pred - y).square().sum().item()
            n += X.size(0)
    return {"loss": total_loss / n, "mae": total_abs / n, "rmse": (total_sq / n) ** 0.5}

### Одна функція запуску

`run_experiment(cfg)` навчає одну конфігурацію з early stopping за validation MAE. Найкращі ваги зберігаються в `CHECKPOINT_DIR/<name>.pth` і відновлюються після навчання.
Оптимізатор — Adam (усі запуски в оригіналі використовували Adam).

In [ ]:
def make_config(name, group, data="raw", arch="base", window_size=WINDOW_SIZE, lr=1e-3, weight_decay=0.0,
                dropout_p=0.0, seed=SEED, batch_size=32, epochs=30, patience=5, min_delta=0.1):
    
    """Повертає словник конфігурації одного запуску. Значення за замовчуванням — протокол розділів 4.1 і 4.2."""
    return {
        "name": name, "group": group, "data": data, "arch": arch, "window_size": window_size,
        "lr": lr, "weight_decay": weight_decay, "dropout_p": dropout_p, "seed": seed,
        "batch_size": batch_size, "epochs": epochs, "patience": patience, "min_delta": min_delta,
    }

In [ ]:
def run_experiment(cfg):
    """Навчає одну конфігурацію з early stopping; повертає рядок результатів і історію за епохами."""
    torch.manual_seed(cfg["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg["seed"])

    train_ds, val_ds, n_channels = get_datasets(cfg["data"], cfg["window_size"])
    generator = torch.Generator().manual_seed(cfg["seed"])
    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"], shuffle=True, generator=generator
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg["batch_size"], shuffle=False
    )

    model = RULCNN(
        in_channels=n_channels,
        window_size=cfg["window_size"],
        dropout_p=cfg["dropout_p"],
        **ARCHITECTURES[cfg["arch"]],
    ).to(DEVICE)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )

    epochs = min(cfg["epochs"], 2) if QUICK else cfg["epochs"]
    ckpt_path = CHECKPOINT_DIR / f"{cfg['name']}.pth"
    history = {"train_loss": [], "val_loss": [], "val_mae": [], "val_rmse": []}
    best_mae, best_epoch, best_state = float("inf"), 0, None
    best_significant_mae, stale = float("inf"), 0

    with mlflow.start_run(run_name=cfg["name"]):
        mlflow.log_params(cfg)
        mlflow.log_param("n_channels", n_channels) 

        for epoch_idx in tqdm(range(epochs), desc=cfg["name"]):
            epoch = epoch_idx + 1 

            train_loss = train_loop(
                train_loader, model, loss_fn, optimizer, DEVICE
            )
            val = eval_loop(val_loader, model, loss_fn, DEVICE)

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val["loss"])
            history["val_mae"].append(val["mae"])
            history["val_rmse"].append(val["rmse"])

            mlflow.log_metrics(
                {
                    "train_loss": train_loss,
                    "val_loss": val["loss"],
                    "val_mae": val["mae"],
                    "val_rmse": val["rmse"],
                },
                step=epoch,
            )

            if val["mae"] < best_mae:
                best_mae, best_epoch = val["mae"], epoch

                best_state = {
                    k: v.cpu().clone() for k, v in model.state_dict().items()
                }
                torch.save(
                    {
                        "model_state_dict": best_state,
                        "best_val_mae": best_mae,
                        "best_epoch": best_epoch,
                        "config": cfg,
                    },
                    ckpt_path,
                )

            # Early stopping
            if val["mae"] < best_significant_mae - cfg["min_delta"]:
                best_significant_mae, stale = val["mae"], 0
            else:
                stale += 1

            print(
                f"Epoch {epoch:02d}/{epochs} | Train MSE: {train_loss:.2f} | Val MAE: {val['mae']:.2f} "
                f"| Val RMSE: {val['rmse']:.2f}"
            )

            if stale >= cfg["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        idx = max(0, best_epoch - 1) if history["val_rmse"] else 0
        best_rmse_val = (
            history["val_rmse"][idx] if history["val_rmse"] else best_mae
        )
        best_mse_val = (
            history["val_loss"][idx] if history["val_loss"] else best_mae**2
        )

        mlflow.log_metrics(
            {
                "best_epoch": best_epoch,
                "best_val_mae": best_mae,
                "best_val_rmse": best_rmse_val,
                "best_val_mse": best_mse_val,
            }
        )

        if best_state is not None:
            model.load_state_dict(best_state)

        row = {
            "experiment": cfg["name"],
            "group": cfg["group"],
            "data": cfg["data"],
            "arch": cfg["arch"],
            "window_size": cfg["window_size"],
            "lr": cfg["lr"],
            "weight_decay": cfg["weight_decay"],
            "dropout_p": cfg["dropout_p"],
            "seed": cfg["seed"],
            "batch_size": cfg["batch_size"],
            "best_epoch": best_epoch,
            "val_mae": best_mae,
            "val_rmse": best_rmse_val,
            "val_mse": best_mse_val,
        }

        print(
            f"{cfg['name']} finished | best epoch {best_epoch} | Val MAE {best_mae:.3f}"
        )

    return row, history

### Список конфігурацій

| Розділ | Дані | Архітектура | Епохи / patience | Batch | Запусків |
|---|---|---|---|---|---|
| 4.1 Експерименти 1–6, A (small), E (nopool) | raw (25) | base / small / nopool | 30 / 5 | 32 | 8 |
| 4.2 Ознаки A, B, C і seeds для A | A (27), B (49), C (66) | nopool | 30 / 5 | 32 | 7 |
| 4.3 Пошук weight decay, потім dropout | A | nopool | 30 / 5 | 32 | 4 + 5 |
| 4.4 Розмір вікна × seeds | A, W = 30 / 50 / 80 | nopool | 60 / 7 | 64 | 12 |

In [21]:
# 4.1: експерименти 1–6 і архітектурні A (small) та E (no pooling) на сирих 25 ознаках
EXP_BASE = [
    make_config("cnn_01_baseline_adam", group="base"),
    make_config("cnn_02_adam_weight_decay", group="base", weight_decay=1e-4),
    make_config("cnn_03_adam_dropout", group="base", dropout_p=0.2),
    make_config("cnn_04_adam_low_lr", group="base", lr=3e-4),
    make_config("cnn_05_adam_dropout_01", group="base", dropout_p=0.1),
    make_config("cnn_06_adam_dropout_03", group="base", dropout_p=0.3),
    make_config("cnn_08_small_channels_dropout_03", group="arch", arch="small", dropout_p=0.3),
    make_config("cnn_12_no_pooling_dropout_03", group="arch", arch="nopool", dropout_p=0.3),
]

# 4.2: набори ознак A, B, C і повтори seed для A (протокол 4.2 — nopool, 30 епох, batch 32)
EXP_FEATURES = [
    make_config("cnn_A_degradation_score", group="features", data="A", arch="nopool", dropout_p=0.3),
    make_config("cnn_B_all_features", group="features", data="B", arch="nopool", dropout_p=0.3),
    make_config("cnn_C_all_features_missing_indicators", group="features", data="C", arch="nopool", dropout_p=0.3),
]
EXP_SEEDS = [
    make_config(f"cnn_A_seed_{s}", group="seeds", data="A", arch="nopool", dropout_p=0.3, seed=s)
    for s in SEEDS
]

# 4.3: пошук weight decay (етап 1); етап 2 (dropout) будується з найкращого weight decay
WD_VALUES = [0.0, 1e-5, 1e-4, 1e-3]
EXP_WD = [
    make_config(f"cnn_A_wd_{wd:g}", group="search_wd", data="A", arch="nopool", weight_decay=wd, dropout_p=0.3)
    for wd in WD_VALUES
]


def dropout_search_configs(best_wd):
    """Конфігурації етапу 2: dropout при найкращому weight decay з етапу 1."""
    return [
        make_config(f"cnn_A_best_wd_{best_wd:g}_dropout_{dp:g}", group="search_dropout", data="A",
                    arch="nopool", weight_decay=best_wd, dropout_p=dp)
        for dp in [0.1, 0.2, 0.3, 0.4, 0.5]
    ]


# 4.4: розмір вікна × seeds (протокол 4.4 — 60 епох, patience 7, batch 64)
WINDOW_SIZES = [30, 50, 80]
EXP_WINDOW = [
    make_config(f"cnn_no_pool_A_w{w}_seed{s}", group="window", data="A", arch="nopool", window_size=w,
                dropout_p=0.3, seed=s, batch_size=64, epochs=60, patience=7)
    for w in WINDOW_SIZES
    for s in SEEDS
]
print("Конфігурацій: 4.1 =", len(EXP_BASE), "| 4.2 =", len(EXP_FEATURES) + len(EXP_SEEDS),
      "| 4.3 =", len(EXP_WD), "+ 5 | 4.4 =", len(EXP_WINDOW))

Конфігурацій: 4.1 = 8 | 4.2 = 7 | 4.3 = 4 + 5 | 4.4 = 12


## 4. Результати

### 4.1 Сирі ознаки (експерименти 1–6, A-small, E)

In [ ]:
df_base = run_experiments(EXP_BASE)
df_base.sort_values("val_mae").round(3)

cnn_01_baseline_adam:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2160.59 | Val MAE: 24.77 | Val RMSE: 32.51
Epoch 03/30 | Train MSE: 1279.25 | Val MAE: 25.83 | Val RMSE: 32.76
Epoch 04/30 | Train MSE: 1189.59 | Val MAE: 24.04 | Val RMSE: 31.14
Epoch 05/30 | Train MSE: 1053.23 | Val MAE: 25.22 | Val RMSE: 33.76
Epoch 06/30 | Train MSE: 886.81 | Val MAE: 23.40 | Val RMSE: 33.22
Epoch 07/30 | Train MSE: 745.69 | Val MAE: 21.72 | Val RMSE: 29.43
Epoch 08/30 | Train MSE: 587.70 | Val MAE: 21.75 | Val RMSE: 29.02
Epoch 09/30 | Train MSE: 463.14 | Val MAE: 22.04 | Val RMSE: 29.36
Epoch 10/30 | Train MSE: 382.40 | Val MAE: 23.38 | Val RMSE: 32.34
Epoch 11/30 | Train MSE: 321.42 | Val MAE: 24.10 | Val RMSE: 32.39
Epoch 12/30 | Train MSE: 269.98 | Val MAE: 22.64 | Val RMSE: 31.39
Early stopping at epoch 12
🏃 View run cnn_01_baseline_adam at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/f374abc243254ba0b25a65fdae399a27
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
cnn_01_baseline_

cnn_02_adam_weight_decay:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2161.56 | Val MAE: 24.81 | Val RMSE: 32.53
Epoch 03/30 | Train MSE: 1280.26 | Val MAE: 25.83 | Val RMSE: 32.76
Epoch 04/30 | Train MSE: 1190.20 | Val MAE: 24.04 | Val RMSE: 31.12
Epoch 05/30 | Train MSE: 1054.90 | Val MAE: 25.22 | Val RMSE: 33.73
Epoch 06/30 | Train MSE: 893.17 | Val MAE: 23.34 | Val RMSE: 33.10
Epoch 07/30 | Train MSE: 753.55 | Val MAE: 21.90 | Val RMSE: 29.65
Epoch 08/30 | Train MSE: 589.92 | Val MAE: 21.93 | Val RMSE: 29.41
Epoch 09/30 | Train MSE: 458.76 | Val MAE: 22.33 | Val RMSE: 29.71
Epoch 10/30 | Train MSE: 373.72 | Val MAE: 23.71 | Val RMSE: 32.63
Epoch 11/30 | Train MSE: 314.24 | Val MAE: 24.29 | Val RMSE: 32.63
Epoch 12/30 | Train MSE: 258.61 | Val MAE: 23.12 | Val RMSE: 32.02
Early stopping at epoch 12
🏃 View run cnn_02_adam_weight_decay at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/d5548ad673664eb8b9a5d440e09272c3
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
cnn_02_adam_

cnn_03_adam_dropout:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2259.46 | Val MAE: 24.28 | Val RMSE: 31.58
Epoch 03/30 | Train MSE: 1386.13 | Val MAE: 24.85 | Val RMSE: 31.57
Epoch 04/30 | Train MSE: 1275.88 | Val MAE: 24.56 | Val RMSE: 32.00
Epoch 05/30 | Train MSE: 1147.91 | Val MAE: 23.63 | Val RMSE: 31.43
Epoch 06/30 | Train MSE: 1011.75 | Val MAE: 23.10 | Val RMSE: 33.24
Epoch 07/30 | Train MSE: 888.68 | Val MAE: 21.09 | Val RMSE: 28.32
Epoch 08/30 | Train MSE: 733.61 | Val MAE: 21.09 | Val RMSE: 28.69
Epoch 09/30 | Train MSE: 620.68 | Val MAE: 21.88 | Val RMSE: 29.32
Epoch 10/30 | Train MSE: 541.79 | Val MAE: 20.98 | Val RMSE: 28.45
Epoch 11/30 | Train MSE: 446.35 | Val MAE: 23.55 | Val RMSE: 31.78
Epoch 12/30 | Train MSE: 398.53 | Val MAE: 23.73 | Val RMSE: 32.89
Epoch 13/30 | Train MSE: 370.28 | Val MAE: 22.84 | Val RMSE: 30.73
Epoch 14/30 | Train MSE: 329.79 | Val MAE: 23.11 | Val RMSE: 31.57
Epoch 15/30 | Train MSE: 312.59 | Val MAE: 22.21 | Val RMSE: 30.45
Early stopping at epoch 15
🏃 View run cnn_03_adam_dropout

... (7 outputs left out)

Epoch 02/30 | Train MSE: 2711.26 | Val MAE: 24.21 | Val RMSE: 31.36
Epoch 03/30 | Train MSE: 1586.61 | Val MAE: 22.97 | Val RMSE: 29.82
Epoch 04/30 | Train MSE: 1486.86 | Val MAE: 22.54 | Val RMSE: 29.56
Epoch 05/30 | Train MSE: 1395.09 | Val MAE: 22.86 | Val RMSE: 29.88
Epoch 06/30 | Train MSE: 1329.00 | Val MAE: 24.20 | Val RMSE: 32.94
Epoch 07/30 | Train MSE: 1302.40 | Val MAE: 21.90 | Val RMSE: 29.07
Epoch 08/30 | Train MSE: 1143.69 | Val MAE: 21.76 | Val RMSE: 28.71
Epoch 09/30 | Train MSE: 1037.57 | Val MAE: 21.15 | Val RMSE: 28.31
Epoch 10/30 | Train MSE: 991.70 | Val MAE: 24.56 | Val RMSE: 32.16
Epoch 11/30 | Train MSE: 935.72 | Val MAE: 21.54 | Val RMSE: 28.43
Epoch 12/30 | Train MSE: 874.92 | Val MAE: 22.40 | Val RMSE: 29.19
Epoch 13/30 | Train MSE: 787.14 | Val MAE: 21.81 | Val RMSE: 29.45
Epoch 14/30 | Train MSE: 790.82 | Val MAE: 22.20 | Val RMSE: 29.36
Early stopping at epoch 14
🏃 View run cnn_08_small_channels_dropout_03 at: https://stage-broom-alkalize.ngrok-free.dev/#/

cnn_12_no_pooling_dropout_03:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2181.06 | Val MAE: 26.20 | Val RMSE: 33.65
Epoch 03/30 | Train MSE: 1453.09 | Val MAE: 23.84 | Val RMSE: 30.96
Epoch 04/30 | Train MSE: 1316.45 | Val MAE: 20.96 | Val RMSE: 28.05
Epoch 05/30 | Train MSE: 1144.69 | Val MAE: 22.02 | Val RMSE: 29.01
Epoch 06/30 | Train MSE: 1061.74 | Val MAE: 22.13 | Val RMSE: 32.08
Epoch 07/30 | Train MSE: 1039.57 | Val MAE: 19.73 | Val RMSE: 27.72
Epoch 08/30 | Train MSE: 941.63 | Val MAE: 20.38 | Val RMSE: 27.91
Epoch 09/30 | Train MSE: 828.89 | Val MAE: 20.43 | Val RMSE: 28.00
Epoch 10/30 | Train MSE: 778.36 | Val MAE: 22.89 | Val RMSE: 30.82
Epoch 11/30 | Train MSE: 711.94 | Val MAE: 21.34 | Val RMSE: 29.39
Epoch 12/30 | Train MSE: 656.76 | Val MAE: 21.30 | Val RMSE: 29.50
Early stopping at epoch 12
🏃 View run cnn_12_no_pooling_dropout_03 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/b5b7014a51c74b948b99f4d56b1fc323
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
cnn_12

,experiment,group,data,arch,window_size,lr,weight_decay,dropout_p,seed,batch_size,best_epoch,val_mae,val_rmse,val_mse
7,cnn_12_no_pooling_dropout_03,arch,raw,nopool,30,0.001,0.0,0.3,42,32,7,19.731,27.910,778.971
5,cnn_06_adam_dropout_03,base,raw,base,30,0.001,0.0,0.3,42,32,7,20.838,28.228,796.837
2,cnn_03_adam_dropout,base,raw,base,30,0.001,0.0,0.2,42,32,10,20.978,31.779,1009.931
6,cnn_08_small_channels_dropout_03,arch,raw,small,30,0.001,0.0,0.3,42,32,9,21.155,32.155,1033.951
0,cnn_01_baseline_adam,base,raw,base,30,0.001,0.0,0.0,42,32,7,21.721,29.017,841.974
1,cnn_02_adam_weight_decay,base,raw,base,30,0.001,0.0,0.0,42,32,7,21.904,29.408,864.832
3,cnn_04_adam_low_lr,base,raw,base,30,0.000,0.0,0.0,42,32,8,21.978,29.430,866.105
4,cnn_05_adam_dropout_01,base,raw,base,30,0.001,0.0,0.1,42,32,7,22.885,30.481,929.096


### 4.2 Набори ознак A, B, C і повтори seed

In [ ]:
df_features = run_experiments(EXP_FEATURES + EXP_SEEDS)
df_features.sort_values("val_mae").round(3)

[A, W=30] train (11304, 30, 27), val (2937, 30, 27)


cnn_A_degradation_score:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2145.18 | Val MAE: 19.73 | Val RMSE: 25.94
Epoch 03/30 | Train MSE: 1257.36 | Val MAE: 18.15 | Val RMSE: 26.08
Epoch 04/30 | Train MSE: 1188.88 | Val MAE: 17.72 | Val RMSE: 23.59
Epoch 05/30 | Train MSE: 1119.10 | Val MAE: 17.32 | Val RMSE: 24.34
Epoch 06/30 | Train MSE: 1025.42 | Val MAE: 17.38 | Val RMSE: 24.53
Epoch 07/30 | Train MSE: 938.60 | Val MAE: 17.60 | Val RMSE: 24.50
Epoch 08/30 | Train MSE: 810.74 | Val MAE: 17.28 | Val RMSE: 24.21
Epoch 09/30 | Train MSE: 684.34 | Val MAE: 19.10 | Val RMSE: 26.85
Epoch 10/30 | Train MSE: 630.14 | Val MAE: 18.14 | Val RMSE: 27.33
Early stopping at epoch 10
🏃 View run cnn_A_degradation_score at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/7d339136066040bd9fbe8fc7c35baba7
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
cnn_A_degradation_score finished | best epoch 8 | Val MAE 17.283
[B, W=30] train (11304, 30, 49), val (2937, 30, 49)


cnn_B_all_features:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 1891.43 | Val MAE: 19.97 | Val RMSE: 27.68
Epoch 03/30 | Train MSE: 1082.68 | Val MAE: 18.95 | Val RMSE: 27.67
Epoch 04/30 | Train MSE: 892.25 | Val MAE: 18.87 | Val RMSE: 26.78
Epoch 05/30 | Train MSE: 694.59 | Val MAE: 21.93 | Val RMSE: 32.22
Epoch 06/30 | Train MSE: 541.71 | Val MAE: 23.82 | Val RMSE: 33.99
Epoch 07/30 | Train MSE: 429.19 | Val MAE: 24.21 | Val RMSE: 35.12
Epoch 08/30 | Train MSE: 386.05 | Val MAE: 24.70 | Val RMSE: 36.33
Early stopping at epoch 8
🏃 View run cnn_B_all_features at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/6e0ff5ceb6504fd4ae849daf02434dac
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
cnn_B_all_features finished | best epoch 4 | Val MAE 18.869
[C, W=30] train (11304, 30, 66), val (2937, 30, 66)


cnn_C_all_features_missing_indicators:   0%|          | 0/30 [00:00<?, ?it/s]

... (6 outputs left out)

Epoch 02/30 | Train MSE: 2080.25 | Val MAE: 19.37 | Val RMSE: 27.45
Epoch 03/30 | Train MSE: 1156.20 | Val MAE: 19.09 | Val RMSE: 26.35
Epoch 04/30 | Train MSE: 1066.22 | Val MAE: 23.02 | Val RMSE: 33.15
Epoch 05/30 | Train MSE: 1015.52 | Val MAE: 21.81 | Val RMSE: 28.79
Epoch 06/30 | Train MSE: 903.46 | Val MAE: 18.29 | Val RMSE: 27.02
Epoch 07/30 | Train MSE: 790.85 | Val MAE: 18.54 | Val RMSE: 27.87
Epoch 08/30 | Train MSE: 704.06 | Val MAE: 18.55 | Val RMSE: 27.78
Epoch 09/30 | Train MSE: 600.45 | Val MAE: 18.02 | Val RMSE: 27.71
Epoch 10/30 | Train MSE: 479.41 | Val MAE: 19.31 | Val RMSE: 28.99
Epoch 11/30 | Train MSE: 455.67 | Val MAE: 20.03 | Val RMSE: 30.22
Epoch 12/30 | Train MSE: 393.70 | Val MAE: 19.20 | Val RMSE: 29.82
Epoch 13/30 | Train MSE: 384.58 | Val MAE: 18.79 | Val RMSE: 27.83
Epoch 14/30 | Train MSE: 362.96 | Val MAE: 21.34 | Val RMSE: 33.54
Early stopping at epoch 14
🏃 View run cnn_A_seed_123 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/09a

cnn_A_seed_2026:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2131.81 | Val MAE: 17.61 | Val RMSE: 23.85
Epoch 03/30 | Train MSE: 1185.39 | Val MAE: 17.65 | Val RMSE: 24.59
Epoch 04/30 | Train MSE: 1130.53 | Val MAE: 18.30 | Val RMSE: 25.11
Epoch 05/30 | Train MSE: 1066.01 | Val MAE: 17.66 | Val RMSE: 25.59
Epoch 06/30 | Train MSE: 946.09 | Val MAE: 17.80 | Val RMSE: 26.02
Epoch 07/30 | Train MSE: 894.61 | Val MAE: 19.42 | Val RMSE: 27.69
Early stopping at epoch 7
🏃 View run cnn_A_seed_2026 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/22aea8b9b0504beb98070808825bb66e
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
cnn_A_seed_2026 finished | best epoch 2 | Val MAE 17.612


,experiment,group,data,arch,window_size,lr,weight_decay,dropout_p,seed,batch_size,best_epoch,val_mae,val_rmse,val_mse
3,cnn_A_seed_7,seeds,A,nopool,30,0.001,0.0,0.3,7,32,3,16.858,24.867,618.345
0,cnn_A_degradation_score,features,A,nopool,30,0.001,0.0,0.3,42,32,8,17.283,26.855,721.188
4,cnn_A_seed_21,seeds,A,nopool,30,0.001,0.0,0.3,21,32,3,17.475,28.034,785.899
6,cnn_A_seed_2026,seeds,A,nopool,30,0.001,0.0,0.3,2026,32,2,17.612,24.592,604.761
5,cnn_A_seed_123,seeds,A,nopool,30,0.001,0.0,0.3,123,32,9,18.020,28.990,840.433
2,cnn_C_all_features_missing_indicators,features,C,nopool,30,0.001,0.0,0.3,42,32,3,18.477,25.373,643.776
1,cnn_B_all_features,features,B,nopool,30,0.001,0.0,0.3,42,32,4,18.869,32.220,1038.155


In [ ]:
# Розкид по seeds для A (seed 42 — базовий запуск cnn_A_degradation_score)
a_seed_names = ["cnn_A_degradation_score"] + [c["name"] for c in EXP_SEEDS]
a_seeds = df_features[df_features["experiment"].isin(a_seed_names)]
print(f"A, {len(a_seeds)} запусків: mean MAE {a_seeds['val_mae'].mean():.3f}, std {a_seeds['val_mae'].std():.3f}, "
      f"min {a_seeds['val_mae'].min():.3f}, max {a_seeds['val_mae'].max():.3f}")

A, 5 запусків: mean MAE 17.449, std 0.427, min 16.858, max 18.020


### 4.3 Пошук weight decay, потім dropout

Етап 1 — weight decay при dropout 0.3. Етап 2 — dropout при найкращому weight decay з етапу 1 (вибір за validation MAE).


In [22]:
df_wd = run_experiments(EXP_WD)
best_wd = df_wd.loc[df_wd["val_mae"].idxmin(), "weight_decay"]
print(f"Найкращий weight decay: {best_wd:g}")

df_dropout = run_experiments(dropout_search_configs(best_wd))
df_dropout.sort_values("val_mae").round(3)

[A, W=30] train (11304, 30, 27), val (2937, 30, 27)


cnn_A_wd_0:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2145.18 | Val MAE: 19.73 | Val RMSE: 25.94
Epoch 03/30 | Train MSE: 1257.34 | Val MAE: 18.15 | Val RMSE: 26.08
Epoch 04/30 | Train MSE: 1188.98 | Val MAE: 17.68 | Val RMSE: 23.55
Epoch 05/30 | Train MSE: 1118.89 | Val MAE: 17.32 | Val RMSE: 24.30
Epoch 06/30 | Train MSE: 1026.87 | Val MAE: 17.39 | Val RMSE: 24.51
Epoch 07/30 | Train MSE: 932.80 | Val MAE: 16.99 | Val RMSE: 23.88
Epoch 08/30 | Train MSE: 801.23 | Val MAE: 17.32 | Val RMSE: 24.23
Epoch 09/30 | Train MSE: 683.59 | Val MAE: 17.66 | Val RMSE: 24.58
Epoch 10/30 | Train MSE: 622.50 | Val MAE: 17.84 | Val RMSE: 26.00
Epoch 11/30 | Train MSE: 535.75 | Val MAE: 18.45 | Val RMSE: 26.60
Epoch 12/30 | Train MSE: 497.36 | Val MAE: 21.44 | Val RMSE: 31.76
Early stopping at epoch 12
🏃 View run cnn_A_wd_0 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/2dba1ad3b54940eea9603e6b13e36303
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
cnn_A_wd_0 finished | bes

cnn_A_wd_1e-05:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2145.41 | Val MAE: 19.71 | Val RMSE: 25.95
Epoch 03/30 | Train MSE: 1259.25 | Val MAE: 18.11 | Val RMSE: 26.06
Epoch 04/30 | Train MSE: 1197.16 | Val MAE: 17.60 | Val RMSE: 23.44
Epoch 05/30 | Train MSE: 1125.58 | Val MAE: 16.94 | Val RMSE: 23.99
Epoch 06/30 | Train MSE: 1026.70 | Val MAE: 17.50 | Val RMSE: 25.15
Epoch 07/30 | Train MSE: 954.71 | Val MAE: 16.74 | Val RMSE: 24.28
Epoch 08/30 | Train MSE: 848.65 | Val MAE: 16.52 | Val RMSE: 24.13
Epoch 09/30 | Train MSE: 730.65 | Val MAE: 17.38 | Val RMSE: 25.66
Epoch 10/30 | Train MSE: 664.53 | Val MAE: 19.23 | Val RMSE: 28.10
Epoch 11/30 | Train MSE: 557.68 | Val MAE: 19.16 | Val RMSE: 27.87
Epoch 12/30 | Train MSE: 498.51 | Val MAE: 20.99 | Val RMSE: 31.20
Epoch 13/30 | Train MSE: 472.10 | Val MAE: 19.70 | Val RMSE: 28.97
Early stopping at epoch 13
🏃 View run cnn_A_wd_1e-05 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/9e2fcc2b377845fc954ec77751dc4cd2
🧪 View experiment at: https://stage-

cnn_A_wd_0.0001:   0%|          | 0/30 [00:00<?, ?it/s]

... (10 outputs left out)

Epoch 02/30 | Train MSE: 2228.48 | Val MAE: 19.94 | Val RMSE: 26.22
Epoch 03/30 | Train MSE: 1363.67 | Val MAE: 18.64 | Val RMSE: 27.07
Epoch 04/30 | Train MSE: 1293.47 | Val MAE: 17.85 | Val RMSE: 23.62
Epoch 05/30 | Train MSE: 1241.71 | Val MAE: 17.29 | Val RMSE: 24.49
Epoch 06/30 | Train MSE: 1113.73 | Val MAE: 17.64 | Val RMSE: 25.29
Epoch 07/30 | Train MSE: 1057.59 | Val MAE: 17.54 | Val RMSE: 25.31
Epoch 08/30 | Train MSE: 942.97 | Val MAE: 17.23 | Val RMSE: 24.48
Epoch 09/30 | Train MSE: 826.85 | Val MAE: 16.87 | Val RMSE: 24.56
Epoch 10/30 | Train MSE: 768.73 | Val MAE: 18.54 | Val RMSE: 27.36
Epoch 11/30 | Train MSE: 658.77 | Val MAE: 20.18 | Val RMSE: 29.78
Epoch 12/30 | Train MSE: 592.62 | Val MAE: 22.34 | Val RMSE: 34.60
Epoch 13/30 | Train MSE: 585.54 | Val MAE: 19.55 | Val RMSE: 29.63
Epoch 14/30 | Train MSE: 503.92 | Val MAE: 22.34 | Val RMSE: 34.33
Early stopping at epoch 14
🏃 View run cnn_A_best_wd_0.001_dropout_0.4 at: https://stage-broom-alkalize.ngrok-free.dev/#/exp

cnn_A_best_wd_0.001_dropout_0.5:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 02/30 | Train MSE: 2384.45 | Val MAE: 20.74 | Val RMSE: 27.20
Epoch 03/30 | Train MSE: 1506.24 | Val MAE: 18.96 | Val RMSE: 27.56
Epoch 04/30 | Train MSE: 1424.47 | Val MAE: 17.39 | Val RMSE: 23.18
Epoch 05/30 | Train MSE: 1406.36 | Val MAE: 17.77 | Val RMSE: 24.66
Epoch 06/30 | Train MSE: 1274.38 | Val MAE: 17.11 | Val RMSE: 24.13
Epoch 07/30 | Train MSE: 1224.40 | Val MAE: 16.22 | Val RMSE: 22.99
Epoch 08/30 | Train MSE: 1132.31 | Val MAE: 17.26 | Val RMSE: 23.68
Epoch 09/30 | Train MSE: 1036.22 | Val MAE: 20.56 | Val RMSE: 29.60
Epoch 10/30 | Train MSE: 977.46 | Val MAE: 18.31 | Val RMSE: 26.53
Epoch 11/30 | Train MSE: 857.44 | Val MAE: 19.09 | Val RMSE: 27.11
Epoch 12/30 | Train MSE: 834.82 | Val MAE: 21.41 | Val RMSE: 31.32
Early stopping at epoch 12
🏃 View run cnn_A_best_wd_0.001_dropout_0.5 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2/runs/cf989a61f704452085dbb7c5a9a6f1df
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/2
c

,experiment,group,data,arch,window_size,lr,weight_decay,dropout_p,seed,batch_size,best_epoch,val_mae,val_rmse,val_mse
0,cnn_A_best_wd_0.001_dropout_0.1,search_dropout,A,nopool,30,0.001,0.001,0.1,42,32,7,15.864,24.599,605.127
4,cnn_A_best_wd_0.001_dropout_0.5,search_dropout,A,nopool,30,0.001,0.001,0.5,42,32,7,16.224,23.678,560.666
2,cnn_A_best_wd_0.001_dropout_0.3,search_dropout,A,nopool,30,0.001,0.001,0.3,42,32,8,16.414,27.146,736.893
3,cnn_A_best_wd_0.001_dropout_0.4,search_dropout,A,nopool,30,0.001,0.001,0.4,42,32,9,16.872,27.358,748.472
1,cnn_A_best_wd_0.001_dropout_0.2,search_dropout,A,nopool,30,0.001,0.001,0.2,42,32,8,16.968,25.877,669.639


### 4.4 Розмір вікна

In [29]:
df_window = run_experiments(EXP_WINDOW)
df_window.sort_values(["window_size", "seed"]).round(3)

cnn_no_pool_A_w30_seed7:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 01/60 | Train MSE: 3001.68 | Val MAE: 17.71 | Val RMSE: 25.63
Epoch 02/60 | Train MSE: 1316.60 | Val MAE: 16.93 | Val RMSE: 23.86
Epoch 03/60 | Train MSE: 1170.09 | Val MAE: 17.12 | Val RMSE: 23.76
Epoch 04/60 | Train MSE: 1088.41 | Val MAE: 18.15 | Val RMSE: 24.45
Epoch 05/60 | Train MSE: 1035.51 | Val MAE: 18.80 | Val RMSE: 26.48
Epoch 06/60 | Train MSE: 995.07 | Val MAE: 19.06 | Val RMSE: 26.56
Epoch 07/60 | Train MSE: 926.50 | Val MAE: 19.12 | Val RMSE: 27.71
Epoch 08/60 | Train MSE: 872.97 | Val MAE: 16.40 | Val RMSE: 22.94
Epoch 09/60 | Train MSE: 772.46 | Val MAE: 20.73 | Val RMSE: 31.98
Epoch 10/60 | Train MSE: 754.81 | Val MAE: 18.18 | Val RMSE: 28.36
Epoch 11/60 | Train MSE: 648.60 | Val MAE: 17.19 | Val RMSE: 25.06
Epoch 12/60 | Train MSE: 602.04 | Val MAE: 17.58 | Val RMSE: 27.16
Epoch 13/60 | Train MSE: 539.88 | Val MAE: 16.48 | Val RMSE: 23.97
Epoch 14/60 | Train MSE: 489.77 | Val MAE: 17.06 | Val RMSE: 24.64
Epoch 15/60 | Train MSE: 432.40 | Val MAE: 17.17 | Val RM

cnn_no_pool_A_w30_seed21:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 01/60 | Train MSE: 2996.04 | Val MAE: 17.83 | Val RMSE: 25.33
Epoch 02/60 | Train MSE: 1386.81 | Val MAE: 17.73 | Val RMSE: 25.55
Epoch 03/60 | Train MSE: 1272.70 | Val MAE: 17.53 | Val RMSE: 23.90
Epoch 04/60 | Train MSE: 1215.01 | Val MAE: 18.31 | Val RMSE: 23.94
Epoch 05/60 | Train MSE: 1178.56 | Val MAE: 17.64 | Val RMSE: 24.77
Epoch 06/60 | Train MSE: 1108.17 | Val MAE: 18.44 | Val RMSE: 24.96
Epoch 07/60 | Train MSE: 1095.79 | Val MAE: 17.93 | Val RMSE: 25.55
Epoch 08/60 | Train MSE: 1044.31 | Val MAE: 17.22 | Val RMSE: 24.72
Epoch 09/60 | Train MSE: 984.23 | Val MAE: 18.05 | Val RMSE: 26.47
Epoch 10/60 | Train MSE: 953.25 | Val MAE: 16.99 | Val RMSE: 24.01
Epoch 11/60 | Train MSE: 939.73 | Val MAE: 20.28 | Val RMSE: 29.80
Epoch 12/60 | Train MSE: 870.27 | Val MAE: 18.06 | Val RMSE: 26.98
Epoch 13/60 | Train MSE: 810.63 | Val MAE: 18.79 | Val RMSE: 27.64
Epoch 14/60 | Train MSE: 727.56 | Val MAE: 18.53 | Val RMSE: 27.41
Epoch 15/60 | Train MSE: 712.42 | Val MAE: 21.98 | Val

cnn_no_pool_A_w30_seed123:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 01/60 | Train MSE: 2768.27 | Val MAE: 19.51 | Val RMSE: 28.49
Epoch 02/60 | Train MSE: 1238.13 | Val MAE: 22.44 | Val RMSE: 31.68
Epoch 03/60 | Train MSE: 1109.77 | Val MAE: 19.19 | Val RMSE: 26.31
Epoch 04/60 | Train MSE: 1071.85 | Val MAE: 18.94 | Val RMSE: 25.93
Epoch 05/60 | Train MSE: 1022.15 | Val MAE: 18.78 | Val RMSE: 26.42
Epoch 06/60 | Train MSE: 956.68 | Val MAE: 19.77 | Val RMSE: 28.67
Epoch 07/60 | Train MSE: 891.32 | Val MAE: 19.22 | Val RMSE: 28.49
Epoch 08/60 | Train MSE: 826.32 | Val MAE: 19.06 | Val RMSE: 28.30
Epoch 09/60 | Train MSE: 719.79 | Val MAE: 22.40 | Val RMSE: 34.39
Epoch 10/60 | Train MSE: 675.41 | Val MAE: 20.18 | Val RMSE: 31.10
Epoch 11/60 | Train MSE: 616.27 | Val MAE: 21.07 | Val RMSE: 32.29
Epoch 12/60 | Train MSE: 555.82 | Val MAE: 18.86 | Val RMSE: 28.86
Early stopping at epoch 12
cnn_no_pool_A_w30_seed123 finished | best epoch 5 | Val MAE 18.775
🏃 View run cnn_no_pool_A_w30_seed123 at: https://stage-broom-alkalize.ngrok-free.dev/#/experiment

... (15 outputs left out)

Epoch 01/60 | Train MSE: 1951.49 | Val MAE: 13.93 | Val RMSE: 20.29
Epoch 02/60 | Train MSE: 696.98 | Val MAE: 13.20 | Val RMSE: 17.90
Epoch 03/60 | Train MSE: 546.45 | Val MAE: 11.26 | Val RMSE: 15.31
Epoch 04/60 | Train MSE: 422.45 | Val MAE: 12.29 | Val RMSE: 16.39
Epoch 05/60 | Train MSE: 356.86 | Val MAE: 11.52 | Val RMSE: 15.80
Epoch 06/60 | Train MSE: 282.80 | Val MAE: 10.36 | Val RMSE: 15.99
Epoch 07/60 | Train MSE: 270.13 | Val MAE: 10.48 | Val RMSE: 15.16
Epoch 08/60 | Train MSE: 239.00 | Val MAE: 9.60 | Val RMSE: 14.54
Epoch 09/60 | Train MSE: 215.35 | Val MAE: 9.37 | Val RMSE: 15.00
Epoch 10/60 | Train MSE: 189.58 | Val MAE: 9.51 | Val RMSE: 14.82
Epoch 11/60 | Train MSE: 185.27 | Val MAE: 9.45 | Val RMSE: 14.78
Epoch 12/60 | Train MSE: 176.37 | Val MAE: 9.45 | Val RMSE: 14.74
Epoch 13/60 | Train MSE: 172.38 | Val MAE: 9.39 | Val RMSE: 14.53
Epoch 14/60 | Train MSE: 170.41 | Val MAE: 9.66 | Val RMSE: 14.85
Epoch 15/60 | Train MSE: 168.85 | Val MAE: 9.19 | Val RMSE: 14.62
Ep

cnn_no_pool_A_w80_seed2026:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 01/60 | Train MSE: 2038.70 | Val MAE: 13.78 | Val RMSE: 20.10
Epoch 02/60 | Train MSE: 745.40 | Val MAE: 12.50 | Val RMSE: 16.52
Epoch 03/60 | Train MSE: 609.04 | Val MAE: 12.06 | Val RMSE: 16.32
Epoch 04/60 | Train MSE: 463.51 | Val MAE: 11.27 | Val RMSE: 14.98
Epoch 05/60 | Train MSE: 348.42 | Val MAE: 10.61 | Val RMSE: 15.91
Epoch 06/60 | Train MSE: 303.75 | Val MAE: 10.72 | Val RMSE: 16.82
Epoch 07/60 | Train MSE: 280.27 | Val MAE: 10.41 | Val RMSE: 15.56
Epoch 08/60 | Train MSE: 244.89 | Val MAE: 9.46 | Val RMSE: 14.87
Epoch 09/60 | Train MSE: 240.53 | Val MAE: 10.59 | Val RMSE: 15.97
Epoch 10/60 | Train MSE: 226.29 | Val MAE: 9.76 | Val RMSE: 15.06
Epoch 11/60 | Train MSE: 222.87 | Val MAE: 11.11 | Val RMSE: 17.26
Epoch 12/60 | Train MSE: 204.86 | Val MAE: 9.58 | Val RMSE: 15.05
Epoch 13/60 | Train MSE: 196.51 | Val MAE: 9.61 | Val RMSE: 15.18
Epoch 14/60 | Train MSE: 197.40 | Val MAE: 10.14 | Val RMSE: 15.63
Epoch 15/60 | Train MSE: 201.89 | Val MAE: 10.21 | Val RMSE: 15.2

,experiment,group,data,arch,window_size,lr,weight_decay,dropout_p,seed,batch_size,best_epoch,val_mae,val_rmse,val_mse
0,cnn_no_pool_A_w30_seed7,window,A,nopool,30,0.001,0.0,0.3,7,64,8,16.400,22.938,526.137
1,cnn_no_pool_A_w30_seed21,window,A,nopool,30,0.001,0.0,0.3,21,64,10,16.987,24.006,576.310
2,cnn_no_pool_A_w30_seed123,window,A,nopool,30,0.001,0.0,0.3,123,64,5,18.775,26.417,697.874
3,cnn_no_pool_A_w30_seed2026,window,A,nopool,30,0.001,0.0,0.3,2026,64,2,17.249,23.820,567.414
4,cnn_no_pool_A_w50_seed7,window,A,nopool,50,0.001,0.0,0.3,7,64,22,12.669,20.829,433.830
5,cnn_no_pool_A_w50_seed21,window,A,nopool,50,0.001,0.0,0.3,21,64,7,15.330,21.519,463.081
6,cnn_no_pool_A_w50_seed123,window,A,nopool,50,0.001,0.0,0.3,123,64,10,13.334,19.208,368.960
7,cnn_no_pool_A_w50_seed2026,window,A,nopool,50,0.001,0.0,0.3,2026,64,10,14.314,20.614,424.950
8,cnn_no_pool_A_w80_seed7,window,A,nopool,80,0.001,0.0,0.3,7,64,9,8.889,13.287,176.539
9,cnn_no_pool_A_w80_seed21,window,A,nopool,80,0.001,0.0,0.3,21,64,6,9.425,13.952,194.645


In [30]:
window_summary = (
    df_window.groupby("window_size")
    .agg(mean_val_mae=("val_mae", "mean"), std_val_mae=("val_mae", "std"),
         best_val_mae=("val_mae", "min"), mean_val_rmse=("val_rmse", "mean"),
         mean_best_epoch=("best_epoch", "mean"))
    .sort_values("mean_val_mae")
)
window_summary.round(3)

,mean_val_mae,std_val_mae,best_val_mae,mean_val_rmse,mean_best_epoch
window_size,,,,,
80,9.240,0.263,8.889,14.182,9.50
50,13.912,1.162,12.669,20.543,12.25
30,17.353,1.012,16.400,24.295,6.25


### Фінальна модель

Найкращий запуск за val MAE в 4.4 копіюється в `DATA_DIR/cnn/cnn_fd001_best.pt`.

In [ ]:
best_run = df_window.loc[df_window["val_mae"].idxmin(), "experiment"]
FINAL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(CHECKPOINT_DIR / f"{best_run}.pth", FINAL_MODEL_PATH)
print(f"Модель ({best_run}) збережено: {FINAL_MODEL_PATH}")

In [ ]:
# Опційно: відновити збережену модель для інференсу (лише якщо файл існує)
if FINAL_MODEL_PATH.exists():
    ckpt = torch.load(FINAL_MODEL_PATH, map_location=DEVICE, weights_only=True)
    cfg = ckpt["config"]
    model = RULCNN(in_channels=cfg.get("in_channels", len(build_split("A")[2])),
                   window_size=cfg.get("window_size", 80), dropout_p=cfg["dropout_p"],
                   **ARCHITECTURES[cfg.get("arch", "nopool")]).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Завантажено модель, best epoch {ckpt['best_epoch']}, val MAE {ckpt['best_val_mae']:.3f}")
else:
    print("Файлу фінальної моделі немає, крок пропущено.")

# Завантаження через MLflow

In [37]:
# 1. Ініціалізуємо порожню модель
from google.colab import files
model = RULCNN(
    in_channels=27,
    window_size=80,
    dropout_p=0.3,
    **ARCHITECTURES["nopool"]
).to(DEVICE)

# 2. Завантажуємо ваги з вже створеного чекпоінта
checkpoint = torch.load(CHECKPOINT_DIR / "cnn_no_pool_A_w80_seed7.pth")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Модель успішно відновлена з локального чекпоінта!")
files.download(CHECKPOINT_DIR / "cnn_no_pool_A_w80_seed7.pth")

Модель успішно відновлена з локального чекпоінта!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Зведена таблиця і графік

Графік validation MAE за епохами для трьох наборів ознак (перевірка перенавчання).

In [ ]:
RESULTS = pd.concat([df_base, df_features, df_wd, df_dropout, df_window], ignore_index=True)
RESULTS[["group", "experiment", "data", "arch", "window_size", "weight_decay", "dropout_p",
         "lr", "best_epoch", "val_mae", "val_rmse", "val_mse"]].round(3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for name in ["cnn_A_degradation_score", "cnn_B_all_features", "cnn_C_all_features_missing_indicators"]:
    hist = HISTORIES[name]
    plt.plot(range(1, len(hist["val_mae"]) + 1), hist["val_mae"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation MAE")
plt.title("CNN: validation MAE за епохами (набори ознак)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Висновки


**1. Сирі 25 ознак (4.1).** Найнижчий MAE дала архітектура без MaxPool з dropout 0.3 (`cnn_12`): 19.762. Далі `cnn_06` (dropout 0.3, 20.712) і `cnn_03` (dropout 0.2, 20.883). Baseline — 21.777. Weight decay 1e-4 (21.925) і менший LR 3e-4 (21.979) не дали покращення. Dropout 0.1 (22.908) гірший за baseline, тож залежність від dropout немонотонна. Кожна конфігурація запущена один раз (seed 42).

**2. Набори ознак (4.2).** Найкращий A: базові + `degradation_score` + індикатор (27 ознак), val MAE 16.210. Для B (49 ознак) 18.894, для C (49 + індикатори) 18.477. Повтори seed для A (7, 21, 123, 2026): 16.873, 17.475, 18.047, 17.612. Разом з seed 42: mean 17.243, std 0.714. B і C запущені лише з seed 42, тож порівнювати їх з A варто з цією оговіркою: різниця B vs C (0.4) менша за розкид seeds, а A краще за B і C приблизно на 2 MAE (порівняння з seed 42).

**3. Weight decay і dropout (4.3).** Weight decay 0, 1e-5, 1e-4, 1e-3 дали 16.184, 16.563, 16.443, 16.683. Dropout 0.1…0.5 при wd 0 дав 17.008, 17.251, 16.368, 17.380, 15.720. Найкращий 15.720 (dropout 0.5) вибрано за validation, тож він оптимістичний. Та сама конфігурація з dropout 0.3 в трьох запусках дала 16.210, 16.184 і 16.368, тому різниці до ~0.3 MAE тут не значущі.

**4. Розмір вікна (4.4).** Val MAE знижується зі збільшенням вікна: W=30 mean 17.346 (std 1.280), W=50 mean 13.856 (std 0.415), W=80 mean 9.178 (std 0.203). Найкращий запуск `cnn_no_pool_A_w80_seed7`, val MAE 9.007, збережений як фінальна модель.

**Найкраща модель для подальшого порівняння:** `cnn_no_pool_A_w80_seed7`, val MAE 9.007. Перед порівнянням з іншими моделями потрібен окремий test-набір FD001 із тими самими ознаками.